# 02 — Feature Engineering
**EduForecast** | OULAD Dataset | Notebook 2 of 6

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DIR, PROCESSED_DIR, FIGURES_DIR, RANDOM_SEED

np.random.seed(RANDOM_SEED)
sns.set_style('whitegrid')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DPI = 150

print('Environment ready.')
print(f'PROCESSED_DIR: {PROCESSED_DIR}')

## 1. Load Raw Data

In [ ]:
print('Loading raw OULAD tables...')

si  = pd.read_csv(RAW_DIR / 'studentInfo.csv')
sr  = pd.read_csv(RAW_DIR / 'studentRegistration.csv')
sv  = pd.read_csv(RAW_DIR / 'studentVle.csv')
ass = pd.read_csv(RAW_DIR / 'assessments.csv')
sa  = pd.read_csv(RAW_DIR / 'studentAssessment.csv')
co  = pd.read_csv(RAW_DIR / 'courses.csv')
vle = pd.read_csv(RAW_DIR / 'vle.csv')

tables = {
    'studentInfo':         si,
    'studentRegistration': sr,
    'studentVle':          sv,
    'assessments':         ass,
    'studentAssessment':   sa,
    'courses':             co,
    'vle':                 vle,
}

print(f'\n{"Table":<25} {"Rows":>10} {"Cols":>6}')
print('-' * 44)
for name, df in tables.items():
    print(f'{name:<25} {df.shape[0]:>10,} {df.shape[1]:>6}')

# Registration-level key: one row per (student × module × presentation)
REG_KEYS = ['id_student', 'code_module', 'code_presentation']
print(f'\nTotal registrations : {len(si):,}')
print(f'Unique students     : {si["id_student"].nunique():,}')
print(f'Students with multiple registrations: {(si.groupby("id_student").size() > 1).sum():,}')

## 2. Target Variable

In [ ]:
print('Creating target variable...')

si = si.copy()
si['is_at_risk'] = (si['final_result'] == 'Withdrawn').astype(int)

n_total    = len(si)
n_risk     = si['is_at_risk'].sum()
n_no_risk  = n_total - n_risk
pct_risk   = n_risk / n_total * 100

print(f'Total registrations : {n_total:,}')
print(f'At-risk (Withdrawn) : {n_risk:,}  ({pct_risk:.1f}%)')
print(f'Not at-risk         : {n_no_risk:,}  ({100 - pct_risk:.1f}%)')
print(f'Class imbalance ratio: {n_no_risk / n_risk:.1f}:1  (not-risk : risk)')

## 3. Static Features

In [ ]:
# ── 3a. Demographic encoding ────────────────────────────────────────────────
print('Encoding demographic features...')

demo = si[REG_KEYS + [
    'gender', 'highest_education', 'age_band', 'imd_band',
    'disability', 'region', 'num_of_prev_attempts',
    'studied_credits', 'is_at_risk',
]].copy()

# Binary
demo['gender']     = (demo['gender'] == 'M').astype(int)
demo['disability'] = (demo['disability'] == 'Y').astype(int)

# Ordinal: highest_education
edu_map = {
    'No Formal quals':              0,
    'Lower Than A Level':           1,
    'A Level or Equivalent':        2,
    'HE Qualification':             3,
    'Post Graduate Qualification':  4,
}
demo['highest_education'] = demo['highest_education'].map(edu_map).fillna(0).astype(int)

# Ordinal: age_band
age_map = {'0-35': 0, '35-55': 1, '55<=': 2}
demo['age_band'] = demo['age_band'].map(age_map).fillna(0).astype(int)

# Ordinal: imd_band — normalise malformed '10-20' → '10-20%', then strip '%'
imd_raw = demo['imd_band'].fillna('').str.replace(r'^10-20$', '10-20%', regex=True)
imd_order = ['0-10%','10-20%','20-30%','30-40%','40-50%',
              '50-60%','60-70%','70-80%','80-90%','90-100%']
imd_map = {v: i for i, v in enumerate(imd_order)}
demo['imd_band'] = imd_raw.map(imd_map)
imd_median = demo['imd_band'].median()
demo['imd_band'] = demo['imd_band'].fillna(imd_median).astype(int)

# One-hot: region
region_dummies = pd.get_dummies(demo['region'], prefix='region', dtype=int)
demo = pd.concat([demo.drop(columns='region'), region_dummies], axis=1)

n_demo_cols = demo.shape[1] - len(REG_KEYS) - 1  # exclude keys and target
print(f'Demographic features encoded: {n_demo_cols} columns')
print(f'imd_band median fill value   : {imd_median:.0f}')

In [ ]:
# ── 3b. VLE aggregate features ──────────────────────────────────────────────
print('Computing VLE aggregate features...')

# Map VLE dates to weeks (clip negative dates to week 0, cap at week 39)
sv_w = sv.copy()
sv_w['week'] = np.ceil(sv_w['date'] / 7).clip(0, 39).astype(int)

# Per-student, per-week clicks
sv_weekly = (
    sv_w.groupby(REG_KEYS + ['week'])['sum_click']
    .sum()
    .reset_index(name='weekly_clicks')
)

# Aggregate to registration level
vle_agg = (
    sv_weekly.groupby(REG_KEYS)
    .agg(
        total_clicks       =('weekly_clicks', 'sum'),
        active_weeks       =('week',          'nunique'),
        max_weekly_clicks  =('weekly_clicks', 'max'),
        mean_weekly_clicks =('weekly_clicks', 'mean'),
        first_active_week  =('week',          'min'),
        last_active_week   =('week',          'max'),
    )
    .reset_index()
)
vle_agg['activity_span'] = vle_agg['last_active_week'] - vle_agg['first_active_week']
vle_agg['mean_weekly_clicks'] = vle_agg['mean_weekly_clicks'].round(2)

VLE_FEAT_COLS = ['total_clicks','active_weeks','max_weekly_clicks',
                 'mean_weekly_clicks','first_active_week','last_active_week','activity_span']
print(f'VLE features: {len(VLE_FEAT_COLS)} columns')
print(f'Registrations with VLE data: {len(vle_agg):,}')

In [ ]:
# ── 3c. Assessment features ──────────────────────────────────────────────────
print('Computing assessment features...')

# Join student submissions with assessment metadata
# assessments.date has 11 nulls (Exams with no fixed date) — drop those rows
ass_clean = ass.dropna(subset=['date']).copy()

sa_full = sa.merge(
    ass_clean[['id_assessment', 'code_module', 'code_presentation',
               'assessment_type', 'date']],
    on='id_assessment',
    how='inner',
)

sa_full['score']       = sa_full['score'].fillna(0)
sa_full['is_late']     = (sa_full['date_submitted'] > sa_full['date']).astype(int)
sa_full['is_submitted']= 1  # all rows in studentAssessment represent a submission

assess_agg = (
    sa_full.groupby(REG_KEYS)
    .agg(
        num_assessments_submitted=('id_assessment',   'count'),
        mean_score               =('score',           'mean'),
        num_late_submissions     =('is_late',         'sum'),
        has_tma                  =('assessment_type', lambda x: int((x == 'TMA').any())),
        has_cma                  =('assessment_type', lambda x: int((x == 'CMA').any())),
    )
    .reset_index()
)
assess_agg['mean_score'] = assess_agg['mean_score'].round(2)

ASSESS_FEAT_COLS = ['num_assessments_submitted','mean_score',
                    'num_late_submissions','has_tma','has_cma']
print(f'Assessment features: {len(ASSESS_FEAT_COLS)} columns')
print(f'Registrations with assessment data: {len(assess_agg):,}')

In [ ]:
# ── 3d. Merge into static feature matrix ────────────────────────────────────
print('Merging into static feature matrix...')

static = (
    demo
    .merge(vle_agg,    on=REG_KEYS, how='left')
    .merge(assess_agg, on=REG_KEYS, how='left')
)

# Fill missing (students with no VLE / no assessment data)
static = static.fillna(0)

# Move is_at_risk to last column
cols = [c for c in static.columns if c != 'is_at_risk'] + ['is_at_risk']
static = static[cols]

feature_cols = [c for c in static.columns
                if c not in REG_KEYS + ['is_at_risk']]

print(f'Static feature matrix shape: {static.shape}')
print(f'Total features: {len(feature_cols)}')
print(f'  Demographic: {n_demo_cols}')
print(f'  VLE aggregate: {len(VLE_FEAT_COLS)}')
print(f'  Assessment: {len(ASSESS_FEAT_COLS)}')
print(f'NaN values: {static.isna().sum().sum()}  (should be 0)')
assert static.isna().sum().sum() == 0, 'NaN values found in static features!'

## 4. Sequential Features (Weekly)

In [ ]:
print('Building sequential feature matrix (weeks 0–39)...')
WEEKS = list(range(40))  # 0 to 39

# ── Step 1: Full registration × week grid ───────────────────────────────────
reg_df = si[REG_KEYS + ['is_at_risk']].drop_duplicates(subset=REG_KEYS)

week_df  = pd.DataFrame({'week': WEEKS})
# Cross-join: every registration × every week
reg_df['_key'] = 1
week_df['_key'] = 1
grid = reg_df.merge(week_df, on='_key').drop(columns='_key')
grid = grid.sort_values(REG_KEYS + ['week']).reset_index(drop=True)

print(f'Full grid: {len(grid):,} rows  ({len(reg_df):,} registrations × {len(WEEKS)} weeks)')

In [ ]:
# ── Step 2: Merge weekly VLE clicks onto the grid ───────────────────────────
seq = grid.merge(
    sv_weekly[REG_KEYS + ['week', 'weekly_clicks']],
    on=REG_KEYS + ['week'],
    how='left',
)
seq['weekly_clicks'] = seq['weekly_clicks'].fillna(0).astype(int)
seq['is_active']     = (seq['weekly_clicks'] > 0).astype(int)
print(f'After VLE merge: {len(seq):,} rows')

In [ ]:
# ── Step 3: Derived temporal features ───────────────────────────────────────
print('Computing cumulative_clicks, click_trend, weeks_since_active...')

seq = seq.sort_values(REG_KEYS + ['week']).reset_index(drop=True)
grp = seq.groupby(REG_KEYS, sort=False)

# cumulative_clicks: running total within each registration
seq['cumulative_clicks'] = grp['weekly_clicks'].cumsum()

# click_trend: change from previous week (0 for first week of each registration)
seq['click_trend'] = grp['weekly_clicks'].diff().fillna(0).astype(int)

# weeks_since_active: weeks elapsed since last active week
# Vectorised: forward-fill last active week within each registration group
seq['_last_active'] = seq['week'].where(seq['is_active'] == 1)
seq['_last_active'] = grp['_last_active'].ffill()
seq['weeks_since_active'] = (seq['week'] - seq['_last_active']).clip(lower=0).fillna(0).astype(int)
seq = seq.drop(columns='_last_active')

print('Temporal features computed.')

In [ ]:
# ── Step 4: Weekly assessment features ──────────────────────────────────────
print('Adding weekly assessment features...')

# Map each assessment to its due week
ass_week = ass_clean.copy()
ass_week['week'] = np.ceil(ass_week['date'] / 7).clip(0, 39).astype(int)

# Join student submissions with assessment week info
sa_weekly = sa.merge(
    ass_week[['id_assessment', 'code_module', 'code_presentation', 'week']],
    on='id_assessment',
    how='inner',
)
sa_weekly['score']          = sa_weekly['score'].fillna(0)
sa_weekly['is_submitted']   = 1

# Aggregate: mean score and max submission flag per (registration, week)
sa_wk_agg = (
    sa_weekly.groupby(REG_KEYS + ['week'])
    .agg(
        assessment_score    =('score',        'mean'),
        assessment_submitted=('is_submitted', 'max'),
    )
    .reset_index()
)
sa_wk_agg['assessment_score']     = sa_wk_agg['assessment_score'].round(2)
sa_wk_agg['assessment_submitted'] = sa_wk_agg['assessment_submitted'].astype(int)

seq = seq.merge(sa_wk_agg, on=REG_KEYS + ['week'], how='left')
seq['assessment_score']     = seq['assessment_score'].fillna(0)
seq['assessment_submitted'] = seq['assessment_submitted'].fillna(0).astype(int)

print(f'Sequential features after assessment merge: {len(seq):,} rows')

In [ ]:
# ── Step 5: Final column order ───────────────────────────────────────────────
SEQ_COLS = REG_KEYS + [
    'week', 'weekly_clicks', 'is_active',
    'cumulative_clicks', 'click_trend', 'weeks_since_active',
    'assessment_score', 'assessment_submitted',
    'is_at_risk',
]
seq = seq[SEQ_COLS]

assert seq.isna().sum().sum() == 0, 'NaN values found in sequential features!'

print(f'Sequential feature matrix shape: {seq.shape}')
print(f'Expected ~{len(reg_df) * 40:,} rows  (actual: {len(seq):,})')
print(f'Features per week: {len(SEQ_COLS) - len(REG_KEYS) - 2}')  # exclude keys, week, target
print(f'NaN values: {seq.isna().sum().sum()}  (should be 0)')

## 5. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

print('Splitting by student ID (80% train / 20% test)...')

# Per-student risk label for stratification
# A student is 'at risk' if ANY of their registrations is Withdrawn
student_risk = (
    si.groupby('id_student')['is_at_risk'].max()
    .reset_index()
)

train_ids, test_ids = train_test_split(
    student_risk['id_student'],
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=student_risk['is_at_risk'],
)
train_ids = set(train_ids)
test_ids  = set(test_ids)

print(f'Train students : {len(train_ids):,}')
print(f'Test students  : {len(test_ids):,}')

# Split static features
train_static = static[static['id_student'].isin(train_ids)].reset_index(drop=True)
test_static  = static[static['id_student'].isin(test_ids)].reset_index(drop=True)

# Split sequential features
train_seq = seq[seq['id_student'].isin(train_ids)].reset_index(drop=True)
test_seq  = seq[seq['id_student'].isin(test_ids)].reset_index(drop=True)

def at_risk_pct(df):
    return df.drop_duplicates(subset=REG_KEYS)['is_at_risk'].mean() * 100

print()
print(f'{"Split":<16} {"Registrations":>14} {"Students":>10} {"At-Risk %":>10}')
print('-' * 54)
print(f'{"Train static":<16} {len(train_static):>14,} {len(train_ids):>10,} {at_risk_pct(train_static):>9.1f}%')
print(f'{"Test static":<16} {len(test_static):>14,} {len(test_ids):>10,} {at_risk_pct(test_static):>9.1f}%')
print(f'{"Train seq":<16} {len(train_seq):>14,} {len(train_ids):>10,} {train_seq.drop_duplicates(REG_KEYS)["is_at_risk"].mean()*100:>9.1f}%')
print(f'{"Test seq":<16} {len(test_seq):>14,} {len(test_ids):>10,} {test_seq.drop_duplicates(REG_KEYS)["is_at_risk"].mean()*100:>9.1f}%')

## 6. Feature Summary

In [ ]:
# ── Feature correlation with is_at_risk ─────────────────────────────────────
print('Computing feature correlations...')

# Use train only — no leakage
corr_cols = [c for c in feature_cols
             if train_static[c].dtype in [np.float64, np.int64, int, float]
             and train_static[c].nunique() > 1]

corr_series = (
    train_static[corr_cols + ['is_at_risk']]
    .corr()['is_at_risk']
    .drop('is_at_risk')
    .abs()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if corr_series.index[i] in VLE_FEAT_COLS
          else '#3498db' if corr_series.index[i] in ASSESS_FEAT_COLS
          else '#2ecc71'
          for i in range(len(corr_series))]

ax.barh(corr_series.index[::-1], corr_series.values[::-1], color=colors[::-1])
ax.set_xlabel('|Pearson r| with is_at_risk', fontsize=12)
ax.set_title('Top 15 Static Features — Correlation with At-Risk Label\n'
             '(red=VLE, blue=Assessment, green=Demographic)', fontsize=13)
ax.set_xlim(0, corr_series.max() * 1.15)

for i, (val, name) in enumerate(zip(corr_series.values[::-1], corr_series.index[::-1])):
    ax.text(val + 0.003, i, f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
fig_path = FIGURES_DIR / '02_feature_importance_corr.png'
fig.savefig(fig_path, dpi=DPI)
plt.close(fig)
print(f'Saved → {fig_path}')

print('\nTop 5 features by correlation with is_at_risk:')
for feat, val in corr_series.head(5).items():
    print(f'  {feat:<28} r = {val:.4f}')

In [ ]:
print('\nStatic feature breakdown:')
print(f'  Total features       : {len(feature_cols)}')
print(f'  Student demographics : {n_demo_cols}')
print(f'  VLE aggregate        : {len(VLE_FEAT_COLS)}')
print(f'  Assessment           : {len(ASSESS_FEAT_COLS)}')

seq_feat_cols = [c for c in SEQ_COLS
                 if c not in REG_KEYS + ['week', 'is_at_risk']]
print('\nSequential feature breakdown:')
print(f'  Total rows           : {len(seq):,}')
print(f'  Weeks per student    : {len(WEEKS)}')
print(f'  Features per week    : {len(seq_feat_cols)}')
print(f'  Train students       : {len(train_ids):,}  ({at_risk_pct(train_static):.1f}% at-risk)')
print(f'  Test students        : {len(test_ids):,}  ({at_risk_pct(test_static):.1f}% at-risk)')

## 7. Save and Verify

In [ ]:
print('Saving processed datasets...')

files = {
    'static_features.csv':     static,
    'sequential_features.csv': seq,
    'train_static.csv':        train_static,
    'test_static.csv':         test_static,
    'train_sequential.csv':    train_seq,
    'test_sequential.csv':     test_seq,
}

for fname, df in files.items():
    out = PROCESSED_DIR / fname
    df.to_csv(out, index=False)
    print(f'  Saved {fname:<30} shape={df.shape}')

print('\nVerifying saved files:')
print(f'{"File":<32} {"Rows":>10} {"Cols":>6}')
print('-' * 52)
for fname in files:
    p = PROCESSED_DIR / fname
    df = pd.read_csv(p, nrows=0)  # read header only for shape check
    full = pd.read_csv(p)
    print(f'{fname:<32} {full.shape[0]:>10,} {full.shape[1]:>6}')
    assert p.exists()

In [ ]:
train_at_risk_pct = at_risk_pct(train_static)
test_at_risk_pct  = at_risk_pct(test_static)

print('=' * 50)
print('FEATURE ENGINEERING COMPLETE')
print('=' * 50)
print(f'Static feature matrix  : {static.shape[0]:,} rows x {static.shape[1]} cols')
print(f'Sequential matrix      : {seq.shape[0]:,} rows x {seq.shape[1]} cols')
print(f'Train students: {len(train_ids):,} | Test students: {len(test_ids):,}')
print(f'At-risk in train: {train_at_risk_pct:.1f}%')
print(f'At-risk in test : {test_at_risk_pct:.1f}%')
print(f'Ready for modeling.')
print('=' * 50)